In [ ]:
import sys
from pathlib import Path
carpeta_pipeline = next(ruta for ruta in [Path.cwd(), Path.cwd() / 'pipeline_ingesta', Path.cwd() / 'notebook/unidad_IV/pipeline_ingesta'] if (ruta / 'auditoria_pipeline.py').is_file())
sys.path.insert(0, str(carpeta_pipeline.resolve()))
from auditoria_pipeline import iniciar_etapa

auditoria_etapa = iniciar_etapa('03_cuarentena', 'invoices')

# 03 - Cuarentena de Invoices

Ejecutar despues de `02_profiling_contratos_Invoices.ipynb` y antes de `04_staging_Invoices.ipynb`.

**Entrada:** `raw.invoices` y errores criticos de `raw.validacion_calidad_invoices`.

**Salida:** `quarantine.invoices` y archivos `cuarentena_invoices_*` en `datos/cuarentena/`.

La clave de exclusion es `(id_raw, lote_id)`. Repetir este notebook no reinserta registros ya aislados. Staging excluye toda cuarentena historica; cambiar su estado no libera automaticamente el registro.

In [1]:
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, BD_DESTINO
from datetime import datetime
import os
import pandas as pd
import json

engine_origen, engine_destino = obtener_engines()

# Rutas
DATOS_DIR = './datos'
CUARENTENA_DIR = os.path.join(DATOS_DIR, 'cuarentena')
os.makedirs(CUARENTENA_DIR, exist_ok=True)

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'Timestamp: {TIMESTAMP}')
print(f'Destino: {BD_DESTINO}')
print(f'Directorio cuarentena: {CUARENTENA_DIR}')

Conexion OK (origen): WideWorldImporters
Conexion OK (destino): WideWorldImportersDW2
Timestamp: 20261103_124855
Destino: WideWorldImportersDW2
Directorio cuarentena: ./datos\cuarentena


## Paso 1: Crear tabla de cuarentena

In [ ]:
from sqlalchemy import text

sql_crear_cuarentena = """
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'quarantine')
    EXEC('CREATE SCHEMA quarantine');

IF OBJECT_ID('quarantine.invoices', 'U') IS NULL
BEGIN
    CREATE TABLE quarantine.invoices (
        id_quarantine       INT PRIMARY KEY IDENTITY(1,1),
        id_raw              INT NOT NULL,
        lote_id             NVARCHAR(50) NOT NULL,
        payload             NVARCHAR(MAX) NOT NULL,
        errores_criticos    NVARCHAR(MAX),
        timestamp_cuarentena DATETIME DEFAULT GETDATE(),
        estado              NVARCHAR(20) DEFAULT 'PENDIENTE',
        motivo_rechazo      NVARCHAR(500),
        timestamp_resolucion DATETIME NULL,
        INDEX IX_raw (id_raw),
        INDEX IX_lote (lote_id),
        INDEX IX_estado (estado)
    );
END;
"""

with engine_destino.begin() as conn:
    conn.execute(text(sql_crear_cuarentena))
print('[OK] Tabla quarantine.invoices verificada/creada')

[OK] Tabla quarantine.invoice_lines creada


## Paso 2: Identificar registros con errores críticos

In [ ]:
sql_errores_criticos = """
SELECT r.id_raw, r.lote_id, r.payload,
    STRING_AGG(
        CAST(CONCAT(v.tipo_error, ': ', v.descripcion) AS NVARCHAR(MAX)),
        ' | '
    ) AS errores_concatenados
FROM raw.invoices r
INNER JOIN raw.validacion_calidad_invoices v
    ON r.id_raw = v.id_raw AND r.lote_id = v.lote_id
WHERE v.es_critico = 1
GROUP BY r.id_raw, r.lote_id, r.payload
"""

df_criticos = pd.read_sql_query(sql_errores_criticos, engine_destino)
print(f'[OK] {len(df_criticos)} cabeceras con errores criticos')
display(df_criticos[['id_raw', 'lote_id']].head())

[OK] Se identificaron 1 registros con errores críticos

Primeros registros problemáticos:


,id_raw,lote_id
0,1,LOTE_20261103_112725


## Paso 3: Guardar en archivo de cuarentena

In [ ]:
if not df_criticos.empty:
    archivo_cuarentena = os.path.join(CUARENTENA_DIR, f'cuarentena_invoices_{TIMESTAMP}.csv')
    df_criticos.to_csv(archivo_cuarentena, index=False, encoding='utf-8')
    archivo_cuarentena_json = os.path.join(CUARENTENA_DIR, f'cuarentena_invoices_{TIMESTAMP}.json')
    with open(archivo_cuarentena_json, 'w', encoding='utf-8') as archivo:
        archivo.write(df_criticos.to_json(orient='records', indent=2, force_ascii=False))
    print(f'[OK] Archivos de cuarentena: {archivo_cuarentena}, {archivo_cuarentena_json}')
else:
    print('[OK] No hay cabeceras con errores criticos')

[OK] Archivo de cuarentena: ./datos\cuarentena\cuarentena_invoice_20261103_124855.csv
Tamaño: 364 bytes
[OK] Archivo JSON: ./datos\cuarentena\cuarentena_invoice_20261103_124855.json


## Paso 4: Insertar en tabla de cuarentena

In [ ]:
sql_insertar_cuarentena = f"""
INSERT INTO quarantine.invoices
    (id_raw, lote_id, payload, errores_criticos, motivo_rechazo)
SELECT r.id_raw, r.lote_id, r.payload, r.errores_concatenados,
       'Errores criticos en validacion de calidad'
FROM ({sql_errores_criticos}) r
WHERE NOT EXISTS (
    SELECT 1 FROM quarantine.invoices q
    WHERE q.id_raw = r.id_raw AND q.lote_id = r.lote_id
)
"""

with engine_destino.begin() as conn:
    insertados = conn.execute(text(sql_insertar_cuarentena)).rowcount
print(f'[OK] {insertados} registros nuevos en quarantine.invoices')

[OK] 1 registros en cuarentena.invoice_lines


## Paso 5: Estadísticas de cuarentena

In [ ]:
sql_stats = """
SELECT COUNT(*) AS total_cuarentena,
       COUNT(DISTINCT lote_id) AS lotes_afectados,
       COUNT(DISTINCT id_raw) AS registros_unicos,
       COUNT(CASE WHEN estado = 'PENDIENTE' THEN 1 END) AS pendiente_resolucion,
       COUNT(CASE WHEN estado = 'RESUELTO' THEN 1 END) AS ya_resueltos
FROM quarantine.invoices
"""
with engine_destino.connect() as conn:
    result = conn.execute(text(sql_stats)).mappings().first()
print('[OK] Estadisticas de cuarentena de Invoices:')
for key, valor in result.items():
    print(f'  {key}: {valor}')

[OK] Estadísticas de cuarentena:
  total_cuarentena: 1
  lotes_afectados: 1
  registros_unicos: 1
  pendiente_resolucion: 1
  ya_resueltos: 0

[OK] Archivos en ./datos\cuarentena:
  - cuarentena_invoice_20261103_124855.csv (364 bytes)
  - cuarentena_invoice_20261103_124855.json (415 bytes)


In [ ]:
auditoria_etapa.finalizar(globals())